# Compresr × LangGraph

You're building a corporate-strategy research graph: retrieve a competitor's profile, feed it to an LLM, hand off to a financial-analyst subagent for valuation work. The retrieved profile is ~45k tokens of mixed-relevance content — most of it irrelevant to the analyst question.

Four groups of integrations cover the four places LangGraph state blows up. Each section runs **without** and **with** compression on the same realistic analyst workload.

| Group | Symbol | What it does |
|---|---|---|
| **A. Graph node** | `make_compresr_node` | Drop a compression step into any `StateGraph`. |
| **B. Middleware** | `CompresrToolMiddleware` | Per-tool output compression. |
|   | `CompresrSummarizationMiddleware` | Rolls old history into one summary message. |
|   | `CompresrPromptMiddleware` | Last-mile prompt cap before the model call. |
| **C. Storage** | `CompresrCheckpointSerializer` | Compresses state blobs before Postgres/Redis. |
|   | `CompresrStore` | Wraps any `BaseStore`; rewrites long fields on write. |
| **D. Multi-agent** | `compresr_handoff_tool` | Supervisor → subagent handoff with compressed payload. |

All examples reuse one company profile so the numbers compare cleanly.

## Install

In [1]:
%pip install -q -e "..[langgraph]" python-dotenv requests openai ipython


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Setup

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / '.env').exists():
        load_dotenv(parent / '.env')
        break

assert os.environ.get('COMPRESR_API_KEY'), 'COMPRESR_API_KEY not set'
assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY not set'
print('API keys loaded.')

API keys loaded.


## Shared setup — competitor research corpus + savings helper

The graph reads a **~45k-token research corpus** stitched live from 12 Wikipedia articles centered on Salesforce (Salesforce + Marc Benioff + Tableau + Slack + MuleSoft + Heroku + adjacent industry context) — the kind of broad-but-noisy dump a research tool returns. `print_savings` builds the $/month table reused across sections.

In [3]:
from _demo_utils import fetch_corpus

CORPUS_TITLES = [
    'Salesforce','Marc Benioff','Tableau Software','Slack Technologies','MuleSoft',
    'Customer relationship management','Heroku','Software industry','Enterprise software',
    'Software as a service','Cloud computing','Service-oriented architecture',
]
COMPANY = 'Salesforce'
QUERY = 'What are Salesforce major acquisitions and how have they shaped its product portfolio?'
article = fetch_corpus(CORPUS_TITLES)
print(f'Research corpus on {COMPANY}: {len(article):,} chars (~{len(article)//4:,} tokens)')

CALLS_PER_DAY = 1_000
DAYS_PER_MONTH = 30

def monthly_cost(tokens: int, price_per_1m: float) -> float:
    return tokens * CALLS_PER_DAY * DAYS_PER_MONTH * price_per_1m / 1_000_000

def print_savings(raw_tokens: int, cmp_tokens: int) -> None:
    prices = {'gpt-5-mini': 0.25, 'gpt-5': 1.25, 'gpt-5.4': 5.00}
    print(f"{'Model':<14}{'Raw $/mo':>14}{'Compresr $/mo':>17}{'Saved $/mo':>14}")
    for name, price in prices.items():
        r = monthly_cost(raw_tokens, price)
        c = monthly_cost(cmp_tokens, price)
        print(f'{name:<14}{r:>14,.2f}{c:>17,.2f}{r - c:>14,.2f}')

Research corpus on Salesforce: 180,171 chars (~45,042 tokens)


## A. Graph node — `make_compresr_node`

**What it does:** a `StateGraph` node that compresses one state field (e.g. `retrieved_text`) and writes the smaller version back.

**When to use it:** custom graphs where you control the topology. Insert it between a retrieval node and the LLM step — every iteration gets the smaller version, KV cache stays warm across turns.

In [ ]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict
from compresr.integrations.langgraph import make_compresr_node
from openai import OpenAI
from IPython.display import display
from _demo_utils import compresr_diff_html

class State(TypedDict, total=False):
    user_question: str
    retrieved_text: str

def retrieve(_state: State) -> dict:
    return {'retrieved_text': article}

def consume(state: State) -> dict:
    return {}

compress = make_compresr_node(
    api_key=os.environ['COMPRESR_API_KEY'],
    context_key='retrieved_text',
    query_key='user_question',
    compression_model='latte_v2',
    target_compression_ratio=0.5,
)

g_raw = StateGraph(State)
g_raw.add_node('retrieve', retrieve).add_node('consume', consume)
g_raw.add_edge(START, 'retrieve').add_edge('retrieve', 'consume').add_edge('consume', END)
raw_final = g_raw.compile().invoke({'user_question': QUERY, 'retrieved_text': ''})

g_cmp = StateGraph(State)
g_cmp.add_node('retrieve', retrieve).add_node('compress', compress).add_node('consume', consume)
g_cmp.add_edge(START, 'retrieve').add_edge('retrieve', 'compress').add_edge('compress', 'consume').add_edge('consume', END)
cmp_final = g_cmp.compile().invoke({'user_question': QUERY, 'retrieved_text': ''})

oai = OpenAI()
SYS = 'You are a precise strategy analyst. Answer only from the provided corpus in 3-4 sentences.'

raw_resp = oai.chat.completions.create(
    model='gpt-5',
    messages=[{'role': 'system', 'content': SYS},
              {'role': 'user', 'content': f"Corpus:\n{raw_final['retrieved_text']}\n\nQuestion: {QUERY}"}],
)
cmp_resp = oai.chat.completions.create(
    model='gpt-5',
    messages=[{'role': 'system', 'content': SYS},
              {'role': 'user', 'content': f"Corpus:\n{cmp_final['retrieved_text']}\n\nQuestion: {QUERY}"}],
)

raw_in = raw_resp.usage.prompt_tokens
cmp_in = cmp_resp.usage.prompt_tokens
saved_pct = (1 - cmp_in / raw_in) * 100

print(f'Without node: {raw_in:>7,} prompt tokens reach the LLM step')
print(f'With node:    {cmp_in:>7,} prompt tokens reach the LLM step   ({saved_pct:.1f}% smaller)')
print(f'$ saved / 1k requests at gpt-5 ($1.25/M input): ${(raw_in - cmp_in) * 1.25 / 1000:.3f}')
print()
print('--- Answer WITHOUT compression ---')
print(raw_resp.choices[0].message.content.strip())
print()
print('--- Answer WITH compression ---')
print(cmp_resp.choices[0].message.content.strip())
print()
print_savings(raw_in, cmp_in)
print()
print('Word-level diff (showing first ~20k chars):')
display(compresr_diff_html(raw_final['retrieved_text'], cmp_final['retrieved_text']))

## B. Middleware — re-exports from LangChain

LangGraph 1.x uses `langchain.agents.create_agent`, so the same middleware works here. They're re-exported through `compresr.integrations.langgraph` for discoverability.

Three layers covering tools, history, and the outbound prompt — see the LangChain notebook (`02_langchain.ipynb`) for the full per-middleware before/after walkthrough. Here's a quick `CompresrPromptMiddleware` demo to confirm the path.

In [5]:
from langchain_core.messages import HumanMessage, ToolMessage
from compresr.integrations.langgraph import (
    CompresrPromptMiddleware,
    CompresrSummarizationMiddleware,
    CompresrToolMiddleware,
)

prompt_mw = CompresrPromptMiddleware(
    api_key=os.environ['COMPRESR_API_KEY'],
    max_tokens=4_000,
    min_tokens=500,
    compression_model='latte_v2',
    query=QUERY,
)

messages = [
    HumanMessage(content=QUERY),
    ToolMessage(content=article, tool_call_id='t1', name='wiki'),
]

class _Req:
    def __init__(self, m): self.messages = m

captured = {}
def handler(r):
    captured['m'] = list(r.messages)
    return 'ok'

prompt_mw.wrap_model_call(_Req(messages), handler)
without = sum(len(m.content) for m in messages) // 4
with_mw = sum(len(m.content) for m in captured['m']) // 4
print(f'Without middleware: {without:>7,} tokens  (over 4k budget)')
print(f'With middleware:    {with_mw:>7,} tokens   ({(1 - with_mw / without) * 100:.1f}% smaller — fits)')
print()
print('All three are re-exported and compose:')
print(f'  - {CompresrToolMiddleware.__name__}')
print(f'  - {CompresrSummarizationMiddleware.__name__}')
print(f'  - {CompresrPromptMiddleware.__name__}')

Without middleware:  45,064 tokens  (over 4k budget)
With middleware:      5,458 tokens   (87.9% smaller — fits)

All three are re-exported and compose:
  - CompresrToolMiddleware
  - CompresrSummarizationMiddleware
  - CompresrPromptMiddleware


## C. Storage — `CompresrCheckpointSerializer` + `CompresrStore`

**What it does:** shrink state *at rest*. `CompresrStore` wraps any `BaseStore` and compresses long string fields on write; `CompresrCheckpointSerializer` compresses large state before LangGraph persists a checkpoint — cutting the bytes written to Postgres/Redis each superstep, then resuming it transparently as text.

**When to use it:** long-running or persisted graphs where retrieved docs / scratchpads get checkpointed every superstep.

> **Field control:** `CompresrStore` sees the whole value dict, so the `fields={...}` allowlist restricts compression to known-large keys. The checkpoint serializer, by contrast, only ever sees one channel value at a time (no field name), so it gates by size via `min_tokens`.

> Compression is **lossy** — the compressed text is what persists and flows back into state on resume.

In [ ]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from compresr.integrations.langgraph import CompresrStore, CompresrCheckpointSerializer

# 1) CompresrStore — wrap any BaseStore; long fields shrink on write.
# The store receives the whole value dict, so the `fields={...}` allowlist works here.
store = CompresrStore(
    InMemoryStore(),
    api_key=os.environ['COMPRESR_API_KEY'],
    fields={'retrieved_text'},
    min_tokens=500,
    compression_model='latte_v2',
    query=QUERY,
)
store.put(('research', COMPANY), 'profile', {'retrieved_text': article, 'note': 'untouched'})
stored = store.get(('research', COMPANY), 'profile').value
si, so = len(article) // 4, len(stored['retrieved_text']) // 4
print('CompresrStore (write-time compression)')
print(f'  retrieved_text: {si:>7,} -> {so:>7,} tokens   ({(1 - so / si) * 100:.1f}% smaller)')
print(f"  non-targeted field passes through untouched: note={stored['note']!r}")
print()

# 2) CompresrCheckpointSerializer — compresses large state before it is
# persisted, and resumes it transparently as text. Checkpointers serialize
# each channel value on its own (no field name visible at the serde layer),
# so gate by size with `min_tokens` rather than `fields`.
serde = CompresrCheckpointSerializer(
    api_key=os.environ['COMPRESR_API_KEY'],
    min_tokens=500,
    compression_model='latte_v2',
    query=QUERY,
)

class CkptState(TypedDict, total=False):
    user_question: str
    retrieved_text: str

ck = StateGraph(CkptState)
ck.add_node('retrieve', lambda _s: {'retrieved_text': article})
ck.add_edge(START, 'retrieve').add_edge('retrieve', END)
ck_app = ck.compile(checkpointer=InMemorySaver(serde=serde))

cfg = {'configurable': {'thread_id': 'demo'}}
ck_app.invoke({'user_question': QUERY, 'retrieved_text': ''}, cfg)
resumed = ck_app.get_state(cfg).values['retrieved_text']  # round-trips through the serde
print('CompresrCheckpointSerializer (state persisted in the checkpoint)')
print(f'  retrieved_text: {len(article) // 4:>7,} -> {len(resumed) // 4:>7,} tokens   ({(1 - len(resumed) / len(article)) * 100:.1f}% smaller)')
print(f'  resumes as {type(resumed).__name__} (usable text, not a wrapper dict)')

## D. Multi-agent — `compresr_handoff_tool`

**What it does:** builds a LangChain tool a supervisor agent can use to hand control to a subagent. The handoff payload — `task_description` and an optional `context` excerpt — is compressed before emitting `Command(goto=..., graph=PARENT)`.

**When to use it:** supervisor / subagent topologies where the supervisor needs to forward a long context excerpt (a RAG snippet, an earlier scratchpad) to a subagent without bloating the subagent's starting prompt.

In [6]:
from compresr.integrations.langgraph import compresr_handoff_tool

financial_handoff = compresr_handoff_tool(
    'financial_analyst',
    api_key=os.environ['COMPRESR_API_KEY'],
    min_tokens=100,
)

tool_call = {
    'name': 'transfer_to_financial_analyst',
    'args': {
        'task_description': f'Estimate the impact of {COMPANY} acquisitions on revenue mix.',
        'context': article,
    },
    'type': 'tool_call',
    'id': 'tc1',
}

cmd = financial_handoff.invoke(tool_call)
before = len(article) // 4
after = len(cmd.update['context']) // 4
print(f'goto agent     : {cmd.goto}   (graph = Command.PARENT)')
print(f'Without handoff: {before:>7,} tokens of context forwarded')
print(f'With handoff:    {after:>7,} tokens of context forwarded   ({(1 - after / before) * 100:.1f}% smaller)')
print()
print('Plug into a supervisor: create_agent(model=..., tools=[financial_handoff, ...])')

goto agent     : financial_analyst   (graph = Command.PARENT)
Without handoff:  45,042 tokens of context forwarded
With handoff:     23,010 tokens of context forwarded   (48.9% smaller)

Plug into a supervisor: create_agent(model=..., tools=[financial_handoff, ...])


## How they compose

Layer compression at the level that matches your bottleneck — most agents need more than one:

- **Inside the graph:** `make_compresr_node` between a retriever and the LLM (group A).
- **Around the model:** any combination of the three middlewares (group B).
- **At rest:** `CompresrCheckpointSerializer` and `CompresrStore` if you persist state with Postgres / Redis (group C).
- **Between agents:** `compresr_handoff_tool` in supervisor / subagent setups (group D).

All four groups use Compresr's `latte_v2` model — query-aware, token-level compression with no LLM round-trip.